In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score


In [89]:
# 데이터 불러오기
df = pd.read_csv('diabetes.csv')

print(df.head())
print(df.shape)
print(df.columns)

   Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin   BMI  \
0            6      148             72             35        0  33.6   
1            1       85             66             29        0  26.6   
2            8      183             64              0        0  23.3   
3            1       89             66             23       94  28.1   
4            0      137             40             35      168  43.1   

   DiabetesPedigreeFunction  Age  Outcome  
0                     0.627   50        1  
1                     0.351   31        0  
2                     0.672   32        1  
3                     0.167   21        0  
4                     2.288   33        1  
(768, 9)
Index(['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin',
       'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome'],
      dtype='str')


In [90]:
# 전처리 전 결측치 확인
missing_before = df.isnull().sum()
missing_before_ratio = df.isnull().mean() * 100

missing_before_df = pd.DataFrame({
    "결측치 개수": missing_before,
    "결측치 비율(%)": missing_before_ratio
})

missing_before_df = missing_before_df[missing_before_df["결측치 개수"] > 0]

print("전처리 전 실제 결측치가 있는 컬럼")
print(missing_before_df)

# 0을 결측치로 볼 컬럼 확인
zero_missing_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

zero_before_list = []

for col in zero_missing_cols:
    if col in df.columns:
        zero_count = (df[col] == 0).sum()
        zero_ratio = (df[col] == 0).mean() * 100

        zero_before_list.append({
            "컬럼명": col,
            "0 개수": zero_count,
            "0 비율(%)": zero_ratio
        })

zero_before_df = pd.DataFrame(zero_before_list)

print("0을 결측치로 처리할 컬럼별 0 개수")
print(zero_before_df)

전처리 전 실제 결측치가 있는 컬럼
Empty DataFrame
Columns: [결측치 개수, 결측치 비율(%)]
Index: []
0을 결측치로 처리할 컬럼별 0 개수
             컬럼명  0 개수    0 비율(%)
0        Glucose     5   0.651042
1  BloodPressure    35   4.557292
2  SkinThickness   227  29.557292
3        Insulin   374  48.697917
4            BMI    11   1.432292


In [91]:
# 데이터 전처리
df = df.copy()

# 불필요한 컬럼 제거
for col in ['id', 'ID', 'Unnamed: 32']:
    if col in df.columns:
        df = df.drop(col, axis=1)

target_col = 'Outcome'

# 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI' 0이 사용됨 <- 이건 결측치 의미 일듯
zero_missing_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
for col in zero_missing_cols:
    if col in df.columns:
        df[col] = df[col].replace(0, np.nan)
        df[col] = df[col].fillna(df[col].median())


# 학습 데이터 / 정답 데이터
X = df.drop(target_col, axis=1)
y = df[target_col].astype(int)

print('X shape =', X.shape)
print('y shape =', y.shape)
print(y.value_counts())

X shape = (768, 8)
y shape = (768,)
Outcome
0    500
1    268
Name: count, dtype: int64


In [92]:
# 전처리 후 결측치 확인
missing_after = df.isnull().sum()
missing_after_ratio = df.isnull().mean() * 100

missing_after_df = pd.DataFrame({
    "처리 후 결측치 개수": missing_after,
    "처리 후 결측치 비율(%)": missing_after_ratio
})

print("전처리 후 결측치 확인")
print(missing_after_df)

전처리 후 결측치 확인
                          처리 후 결측치 개수  처리 후 결측치 비율(%)
Pregnancies                         0             0.0
Glucose                             0             0.0
BloodPressure                       0             0.0
SkinThickness                       0             0.0
Insulin                             0             0.0
BMI                                 0             0.0
DiabetesPedigreeFunction            0             0.0
Age                                 0             0.0
Outcome                             0             0.0


In [93]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import minmax_scale


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.1,
    random_state=42,
    stratify=y
)

# # StandardScaler LogicGate 정확도 => 0.7272727272727273
# scaler = StandardScaler()
# X_train_scaled = scaler.fit_transform(X_train)
# X_test_scaled = scaler.transform(X_test)

# Min-Max 스케일링 LogicGate 정확도 => 0.7792207792207793
X_train_scaled = minmax_scale(X_train)
X_test_scaled = minmax_scale(X_test)

print("X_train shape =", X_train.shape)
print("X_test shape =", X_test.shape)
print("y_train shape =", y_train.shape)
print("y_test shape =", y_test.shape)



X_train shape = (691, 8)
X_test shape = (77, 8)
y_train shape = (691,)
y_test shape = (77,)


In [94]:
# 시그모이드 함수
def sigmoid(x):
    x = np.clip(x, -500, 500)
    return 1. / (1. + np.exp(-x))

In [110]:
class LogicGate:
    def __init__(self, gate_name, xdata, tdata):  
        self.name = gate_name
        self.xdata = np.array(xdata)
        self.tdata = np.array(tdata).reshape(-1, 1)

        self.input_nodes = self.xdata.shape[1]
        self.hidden_nodes1 = 16
        self.hidden_nodes2 = 8
        self.output_nodes = 1

        # 입력층 -> 첫 번째 은닉층
        self.W2 = np.random.randn(self.input_nodes, self.hidden_nodes1) * np.sqrt(1 / self.input_nodes)
        self.b2 = np.zeros(self.hidden_nodes1)

        # 첫 번째 은닉층 -> 두 번째 은닉층
        self.W3 = np.random.randn(self.hidden_nodes1, self.hidden_nodes2) * np.sqrt(1 / self.hidden_nodes1)
        self.b3 = np.zeros(self.hidden_nodes2)
        
        # 두 번째 은닉층 -> 출력층
        self.W4 = np.random.randn(self.hidden_nodes2, self.output_nodes) * np.sqrt(1 / self.hidden_nodes2)
        self.b4 = np.zeros(self.output_nodes)

        self.learning_rate = 1e-2

    def forward(self, x):
        self.z2 = np.dot(x, self.W2) + self.b2
        self.a2 = sigmoid(self.z2)

        self.z3 = np.dot(self.a2, self.W3) + self.b3
        self.a3 = sigmoid(self.z3)

        self.z4 = np.dot(self.a3, self.W4) + self.b4
        self.y = sigmoid(self.z4)

        return self.y

    def loss_val(self):
        delta = 1e-7
        y = self.forward(self.xdata)
        return -np.mean(self.tdata * np.log(y + delta) + (1 - self.tdata) * np.log(1 - y + delta))

    def train(self):
        print('Initial loss value =', self.loss_val())

        for step in range(200001):
            # forward
            y = self.forward(self.xdata)
            m = self.xdata.shape[0]

            # backpropagation
            dz4 = (y - self.tdata) / m
            dW4 = np.dot(self.a3.T, dz4)
            db4 = np.sum(dz4, axis=0)

            da3 = np.dot(dz4, self.W4.T)
            dz3 = da3 * self.a3 * (1 - self.a3)
            dW3 = np.dot(self.a2.T, dz3)
            db3 = np.sum(dz3, axis=0)

            da2 = np.dot(dz3, self.W3.T)
            dz2 = da2 * self.a2 * (1 - self.a2)
            dW2 = np.dot(self.xdata.T, dz2)
            db2 = np.sum(dz2, axis=0)

            # weight update
            self.W4 -= self.learning_rate * dW4
            self.b4 -= self.learning_rate * db4
            self.W3 -= self.learning_rate * dW3
            self.b3 -= self.learning_rate * db3
            self.W2 -= self.learning_rate * dW2
            self.b2 -= self.learning_rate * db2

            if step % 10000 == 0:
                train_pred = self.predict(self.xdata)
                train_acc = accuracy_score(self.tdata.flatten(), train_pred)
                print('step =', step, 'loss value =', self.loss_val(), 'train accuracy =', train_acc)

    def predict_proba(self, input_data):
        input_data = np.array(input_data)
        return self.forward(input_data)

    def predict(self, input_data):
        y = self.predict_proba(input_data)
        return (y >= 0.5).astype(int).flatten()

    def accuracy(self, test_xdata, test_tdata):
        pred = self.predict(test_xdata)
        accuracy_val = accuracy_score(test_tdata, pred)
        return accuracy_val

In [111]:
# (1) LogicGate 클래스를 활용한 분류 모델
LogicGate_obj = LogicGate('DIABETES_CLASSIFICATION', X_train_scaled, y_train)
LogicGate_obj.train()

logic_pred = LogicGate_obj.predict(X_test_scaled)
logic_accuracy = accuracy_score(y_test, logic_pred)

print('----------------------------------------')
print('LogicGate 정확도 =>', logic_accuracy)

Initial loss value = 0.711871268034322
step = 0 loss value = 0.7105873351622022 train accuracy = 0.34876989869753977
step = 10000 loss value = 0.6377632135852916 train accuracy = 0.6512301013024602
step = 20000 loss value = 0.5941831613406728 train accuracy = 0.6512301013024602
step = 30000 loss value = 0.48722101468784873 train accuracy = 0.768451519536903
step = 40000 loss value = 0.4617265843517521 train accuracy = 0.768451519536903
step = 50000 loss value = 0.4578473584533087 train accuracy = 0.7800289435600579
step = 60000 loss value = 0.45654490823422866 train accuracy = 0.7829232995658466
step = 70000 loss value = 0.4558497569971177 train accuracy = 0.7814761215629522
step = 80000 loss value = 0.45537850872365976 train accuracy = 0.7814761215629522
step = 90000 loss value = 0.4550155698734279 train accuracy = 0.7785817655571635
step = 100000 loss value = 0.4547116048084571 train accuracy = 0.7771345875542692
step = 110000 loss value = 0.4544391497807732 train accuracy = 0.777134

In [112]:
# (2) 케라스 라이브러리를 활용한 분류 모델

import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
import tensorflow as tf
from sklearn.metrics import accuracy_score

# Keras 모델 생성
# 가장 좋은 히든층 구조: [16]
keras_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train_scaled.shape[1],)),
    
    tf.keras.layers.Dense(16, activation='relu'),
    
    tf.keras.layers.Dense(1, activation='sigmoid')
])

keras_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='mse',
    metrics=['accuracy']
)

# 모델 구조 확인
keras_model.summary()

# 과적합 방지를 위한 EarlyStopping
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=20,
    restore_best_weights=True
)

# 모델 학습
history = keras_model.fit(
    X_train_scaled,
    y_train,
    epochs=1000,
    batch_size=64,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
)

# Keras evaluate()로 평가
loss, accuracy = keras_model.evaluate(X_test_scaled, y_test, verbose=0)

print("Keras loss =>", loss)
print("Keras evaluate 정확도 =>", accuracy)

# accuracy_score()로 정확도 평가
y_pred_proba = keras_model.predict(X_test_scaled)

y_pred = []

for prob in y_pred_proba:
    if prob[0] >= 0.5:
        y_pred.append(1)
    else:
        y_pred.append(0)

y_pred = np.array(y_pred)

keras_accuracy = accuracy_score(y_test, y_pred)

print("Keras accuracy_score 정확도 =>", keras_accuracy)

Model: "sequential_22"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_79 (Dense)            (None, 16)                144       
                                                                 
 dense_80 (Dense)            (None, 1)                 17        
                                                                 
Total params: 161 (644.00 Byte)
Trainable params: 161 (644.00 Byte)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


Epoch 1/1000
10/10 [==============================] - 0s 12ms/step - loss: 0.2502 - accuracy: 0.5137 - val_loss: 0.2465 - val_accuracy: 0.5714
Epoch 2/1000
10/10 [==============================] - 0s 3ms/step - loss: 0.2450 - accuracy: 0.6087 - val_loss: 0.2376 - val_accuracy: 0.6714
Epoch 3/1000
10/10 [==============================] - 0s 3ms/step - loss: 0.2408 - accuracy: 0.6312 - val_loss: 0.2318 - val_accuracy: 0.7143
Epoch 4/1000
10/10 [==============================] - 0s 3ms/step - loss: 0.2386 - accuracy: 0.6393 - val_loss: 0.2274 - val_accuracy: 0.7286
Epoch 5/1000
10/10 [==============================] - 0s 3ms/step - loss: 0.2363 - accuracy: 0.6393 - val_loss: 0.2251 - val_accuracy: 0.7286
Epoch 6/1000
10/10 [==============================] - 0s 3ms/step - loss: 0.2348 - accuracy: 0.6393 - val_loss: 0.2228 - val_accuracy: 0.7286
Epoch 7/1000
10/10 [==============================] - 0s 3ms/step - loss: 0.2331 - accuracy: 0.6393 - val_loss: 0.2212 - val_accuracy: 0.7286
Epoch